# 📈 Notebook 3 — Exploratory Data Analysis (EDA)

**Project:** Customer Churn Prediction  
**Objective:** Visualise key patterns in the cleaned dataset to understand what drives customer churn.

**Plots generated:**
1. Churn distribution (pie + bar)
2. Contract type vs Churn
3. Tenure vs Churn (histogram + KDE)
4. Monthly Charges distribution
5. Correlation heatmap
6. Internet Service vs Churn
7. Payment Method vs Churn
8. Senior Citizen vs Churn
9. Total Charges vs Tenure (scatter)
10. Churn by Tenure Group

All plots are saved to `outputs/plots/`.

---

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.join('..', 'src'))
from utils import load_dataset, ensure_dir
from preprocessing import clean_data, engineer_features

# Plot settings
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PLOTS_DIR = os.path.join('..', 'outputs', 'plots')
ensure_dir(PLOTS_DIR)

print('Libraries loaded ✅')

In [ ]:
# Load and clean dataset
DATA_PATH = os.path.join('..', 'data', 'customer_churn.csv')
df_raw = load_dataset(DATA_PATH)
df = clean_data(df_raw)
df = engineer_features(df)
print(f'Working dataset shape: {df.shape}')

## 3.1  Churn Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
churn_counts = df['Churn'].value_counts()
colors = ['#2196F3', '#F44336']
bars = axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Churn Distribution', fontweight='bold')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Number of Customers')
for bar, val in zip(bars, churn_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=11)

# Pie chart
axes[1].pie(churn_counts.values, labels=churn_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0, 0.05),
            textprops={'fontsize': 12})
axes[1].set_title('Churn Proportion', fontweight='bold')

plt.suptitle('Customer Churn Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '01_churn_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('⚠  Class imbalance: ~73% No Churn vs ~27% Churn')

## 3.2  Contract Type vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar
contract_churn = df.groupby(['Contract', 'Churn']).size().unstack(fill_value=0)
contract_churn.plot(kind='bar', stacked=True, ax=axes[0], color=['#2196F3', '#F44336'],
                    edgecolor='white')
axes[0].set_title('Contract Type vs Churn (Count)', fontweight='bold')
axes[0].set_xlabel('Contract Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(['No Churn', 'Churn'])

# Percentage bar
contract_pct = contract_churn.div(contract_churn.sum(axis=1), axis=0) * 100
contract_pct.plot(kind='bar', stacked=True, ax=axes[1], color=['#2196F3', '#F44336'],
                  edgecolor='white')
axes[1].set_title('Contract Type vs Churn (Proportion %)', fontweight='bold')
axes[1].set_xlabel('Contract Type')
axes[1].set_ylabel('Percentage (%)')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(['No Churn', 'Churn'], loc='upper right')

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '02_contract_vs_churn.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📌 Insight: Month-to-month contracts have by far the highest churn rate.')

## 3.3  Tenure vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE plot by churn class
for label, color in zip(['No', 'Yes'], ['#2196F3', '#F44336']):
    subset = df[df['Churn'] == label]
    axes[0].hist(subset['tenure'], bins=30, alpha=0.6, color=color, label=f'Churn={label}',
                 edgecolor='white', density=True)
axes[0].set_title('Tenure Distribution by Churn', fontweight='bold')
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Box plot
df.boxplot(column='tenure', by='Churn', ax=axes[1],
           boxprops=dict(color='#2196F3'),
           medianprops=dict(color='red', linewidth=2),
           whiskerprops=dict(color='gray'),
           capprops=dict(color='gray'))
axes[1].set_title('Tenure Box Plot by Churn', fontweight='bold')
axes[1].set_xlabel('Churn')
axes[1].set_ylabel('Tenure (months)')
plt.suptitle('')  # Remove default suptitle from boxplot

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '03_tenure_vs_churn.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📌 Insight: Churned customers tend to have shorter tenures (early-tenure churn risk).')

## 3.4  Monthly Charges Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
axes[0].hist(df['MonthlyCharges'], bins=40, color='#9C27B0', edgecolor='white', alpha=0.8)
axes[0].axvline(df['MonthlyCharges'].mean(), color='red', linestyle='--',
                 label=f'Mean: ${df["MonthlyCharges"].mean():.1f}')
axes[0].axvline(df['MonthlyCharges'].median(), color='orange', linestyle='--',
                 label=f'Median: ${df["MonthlyCharges"].median():.1f}')
axes[0].set_title('Monthly Charges Distribution', fontweight='bold')
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].set_ylabel('Count')
axes[0].legend()

# By churn status
for label, color in zip(['No', 'Yes'], ['#2196F3', '#F44336']):
    subset = df[df['Churn'] == label]
    axes[1].hist(subset['MonthlyCharges'], bins=30, alpha=0.6,
                 color=color, label=f'Churn={label}', edgecolor='white')
axes[1].set_title('Monthly Charges by Churn Status', fontweight='bold')
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '04_monthly_charges_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📌 Insight: Customers who churn tend to have higher monthly charges.')

## 3.5  Correlation Heatmap

In [ ]:
# Encode Churn as numeric for correlation
df_corr = df.copy()
df_corr['Churn_num'] = (df_corr['Churn'] == 'Yes').astype(int)

numeric_cols = df_corr.select_dtypes(include=np.number).columns.tolist()
corr_matrix  = df_corr[numeric_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # show lower triangle only
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    square=True,
    vmin=-1,
    vmax=1,
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '05_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📌 Insight: tenure is negatively correlated with Churn; MonthlyCharges is positively correlated.')

## 3.6  Internet Service vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

internet_churn = df.groupby(['InternetService', 'Churn']).size().unstack(fill_value=0)
internet_pct   = internet_churn.div(internet_churn.sum(axis=1), axis=0) * 100

internet_churn.plot(kind='bar', ax=axes[0], color=['#2196F3', '#F44336'], edgecolor='white')
axes[0].set_title('Internet Service vs Churn (Count)', fontweight='bold')
axes[0].set_xlabel('Internet Service')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(['No Churn', 'Churn'])

internet_pct.plot(kind='bar', ax=axes[1], color=['#2196F3', '#F44336'], edgecolor='white')
axes[1].set_title('Internet Service vs Churn (Proportion %)', fontweight='bold')
axes[1].set_xlabel('Internet Service')
axes[1].set_ylabel('Percentage (%)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(['No Churn', 'Churn'])

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '06_internet_service_vs_churn.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📌 Insight: Fiber optic internet users churn at significantly higher rates.')

## 3.7  Payment Method vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

pay_churn = df.groupby(['PaymentMethod', 'Churn']).size().unstack(fill_value=0)
pay_pct   = pay_churn.div(pay_churn.sum(axis=1), axis=0) * 100

pay_churn.plot(kind='barh', ax=axes[0], color=['#2196F3', '#F44336'], edgecolor='white')
axes[0].set_title('Payment Method vs Churn (Count)', fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].legend(['No Churn', 'Churn'])

pay_pct.plot(kind='barh', ax=axes[1], color=['#2196F3', '#F44336'], edgecolor='white')
axes[1].set_title('Payment Method vs Churn (Proportion %)', fontweight='bold')
axes[1].set_xlabel('Percentage (%)')
axes[1].legend(['No Churn', 'Churn'])

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '07_payment_method_vs_churn.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📌 Insight: Electronic check users have the highest churn rate.')

## 3.8  Senior Citizen vs Churn

In [ ]:
senior_churn = df.groupby(['SeniorCitizen', 'Churn']).size().unstack(fill_value=0)
senior_churn.index = ['Non-Senior', 'Senior']
senior_pct = senior_churn.div(senior_churn.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
senior_churn.plot(kind='bar', ax=axes[0], color=['#2196F3', '#F44336'], edgecolor='white')
axes[0].set_title('Senior Citizen vs Churn (Count)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(['No Churn', 'Churn'])

senior_pct.plot(kind='bar', ax=axes[1], color=['#2196F3', '#F44336'], edgecolor='white')
axes[1].set_title('Senior Citizen vs Churn (Proportion %)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=0)
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(['No Churn', 'Churn'])

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '08_senior_citizen_vs_churn.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📌 Insight: Senior citizens have a ~41% churn rate vs ~24% for non-seniors.')

## 3.9  Total Charges vs Tenure (Scatter)

In [ ]:
plt.figure(figsize=(10, 6))
for label, color, marker in zip(['No', 'Yes'], ['#2196F3', '#F44336'], ['o', 'x']):
    subset = df[df['Churn'] == label]
    plt.scatter(
        subset['tenure'],
        subset['TotalCharges'],
        c=color,
        alpha=0.3,
        s=15,
        marker=marker,
        label=f'Churn={label}'
    )
plt.title('Total Charges vs Tenure by Churn Status', fontweight='bold')
plt.xlabel('Tenure (months)')
plt.ylabel('Total Charges ($)')
plt.legend(markerscale=2)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '09_total_charges_vs_tenure.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📌 Insight: Churned customers cluster in the lower-left (low tenure, lower total charges).')

## 3.10  Churn Rate by Tenure Group

In [ ]:
tenure_churn = df.groupby('tenure_group', observed=True)['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
tenure_churn.columns = ['Tenure Group', 'Churn Rate (%)']

plt.figure(figsize=(10, 5))
bars = plt.bar(
    tenure_churn['Tenure Group'].astype(str),
    tenure_churn['Churn Rate (%)'],
    color=plt.cm.RdYlGn_r(tenure_churn['Churn Rate (%)'] / 100),
    edgecolor='white'
)
for bar, val in zip(bars, tenure_churn['Churn Rate (%)']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', fontsize=10)
plt.title('Churn Rate by Tenure Group', fontweight='bold')
plt.xlabel('Tenure (months)')
plt.ylabel('Churn Rate (%)')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, '10_churn_by_tenure_group.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📌 Insight: Customers in the 0-12 month group have the highest churn rate.')

---
## EDA Summary — Key Insights

| Feature | Insight |
|---------|----------|
| Contract | Month-to-month → highest churn |
| Tenure | Short tenure → much higher churn risk |
| MonthlyCharges | Higher charges → higher churn |
| InternetService | Fiber optic → significantly more churn |
| PaymentMethod | Electronic check → highest churn |
| SeniorCitizen | Seniors churn ~41% vs ~24% non-senior |
| OnlineSecurity/TechSupport | No add-ons → higher churn |

➡  **Next:** `04_model_training.ipynb` — encode features and train multiple ML models.